# Step 1: Understanding Text Noise

In [1]:
import pandas as pd
import numpy as np

import re
import string

import nltk
import spacy

from nltk.corpus import stopwords

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv(
    "../data/processed/imdb_cleaned.csv"
)

print(df.shape)

(49582, 2)


In [3]:
df["review"].sample(5).values

<ArrowStringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [4]:
sample_review = df["review"][0]

print(sample_review)

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac

# Step 2: Lowercasing

In [5]:
df["clean_review"] = df["review"]

In [6]:
df[["review", "clean_review"]].head()

,review,clean_review
0,One of the other reviewers has mentioned that ...,One of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,A wonderful little production. <br /><br />The...
2,I thought this was a wonderful way to spend ti...,I thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,Basically there's a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...","Petter Mattei's ""Love in the Time of Money"" is..."


# review         → Original Data (Never Touch)
# clean_review   → Working Copy (Apply Cleaning)

In [7]:
df["clean_review"] = df["clean_review"].str.lower()

In [8]:
print(df["review"][0][:200])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo


In [9]:
print(df["clean_review"][0][:200])

one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me abo


# Step 2 Findings

### Lowercasing

Text was converted to lowercase to ensure consistency.

Examples:

Movie → movie

MOVIE → movie

movie → movie

This reduces vocabulary size and improves model performance.

# Step 3: Remove HTML Tags 

In [10]:
from bs4 import BeautifulSoup

In [11]:
def remove_html_tags(text):
    return BeautifulSoup(text, "html.parser").get_text()

In [12]:
df["clean_review"] = df["clean_review"].apply(
    remove_html_tags
)

In [13]:
print(df["review"][0][:300])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru


In [14]:
print(df["clean_review"][0][:300])

one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this 


# Step 3 Findings

### HTML Tag Removal

HTML tags such as:

- <br />
- <br /><br />

were removed successfully.

These tags do not contribute to sentiment and are considered textual noise.

### Conclusion

The review text is now cleaner and more suitable for NLP processing.

In [15]:
df["clean_review"].str.contains(
    "http|www",
    case=False,
    regex=True
).sum()

np.int64(243)

# Step 4 – URL Removal

In [16]:
url_reviews = df[
    df["clean_review"].str.contains(
        "http|www",
        case=False,
        regex=True
    )
]

url_reviews["clean_review"].head(5).values

<ArrowStringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

# Step 4: URL Removal

URLs are considered noise for sentiment analysis.

We remove them because they usually do not contribute to determining sentiment.

In [17]:
def remove_urls(text):
    pattern = r'https?://\S+|www\.\S+'
    return re.sub(pattern, '', text)

In [18]:
df["clean_review"] = df["clean_review"].apply(
    remove_urls
)

In [19]:
df["clean_review"].str.contains(
    "http|www",
    case=False,
    regex=True
).sum()

np.int64(50)

In [20]:
pattern = r'https?://\S+|www\.\S+'

In [21]:
remaining_urls = df[
    df["clean_review"].str.contains(
        "http|www",
        case=False,
        regex=True
    )
]

remaining_urls["clean_review"].head(10).tolist()

['first of all, let\'s get a few things straight here: a) i am an anime fan- always has been as a matter of fact (i used to watch speed racer all the time in preschool). b) i do like several b-movies because they\'re hilarious. c) i like the godzilla movies- a lot.moving on, when the movie first comes on, it seems like it\'s going to be your usual b-movie, down to the crappy fx, but all a sudden- boom! the anime comes on! this is when the movie goes wwwaaaaayyyyy downhill.the animation is very bad & cheap, even worse than what i remember from speed racer, for crissakes! in fact, it\'s so cheap, one of the few scenes from the movie i "vividly" remember is when a bunch of kids run out of a school... & it\'s the same kids over & over again! the fx are terrible, too; the dinosaurs look worse than godzilla. in addition, the transition to live action to animation is unorganized, the dialogue & voices(especially the english dub that i viewed) was horrid & i was begging my dad to take the tape

In [22]:
df[
    df["clean_review"].str.contains(
        r'https?://|www\.',
        case=False,
        regex=True
    )
]

,review,sentiment,clean_review
3534,It is finally coming out. The first season wil...,positive,it is finally coming out. the first season wil...
3940,"OK, to start with, this movie was not at all l...",negative,"ok, to start with, this movie was not at all l..."
11223,"Seriously, all these Satan comes to Earth movi...",negative,"seriously, all these satan comes to earth movi..."
11282,I can't believe I'm wasting my time with a com...,negative,i can't believe i'm wasting my time with a com...
19423,POSSIBLE SPOILERS --<br /><br />I love Dennis ...,negative,possible spoilers --i love dennis quaid and i ...
30411,"It's a kinder, gentler Cyborg movie with a lov...",negative,"it's a kinder, gentler cyborg movie with a lov..."
40931,"I really wanted to like this movie, but it was...",negative,"i really wanted to like this movie, but it was..."


In [25]:
remaining_urls = df[
    df["clean_review"].str.contains(
        r'https?://|www\.',
        case=False,
        regex=True
    )
]

remaining_urls["clean_review"].head(10).tolist()

['it is finally coming out. the first season will be available march 2007. it is currently airing on abc family from 4-5 pm eastern time monday through friday. the last episode will air on december 19th at 4:30. i missed it the first 100 times around. i wish i could buy the whole series right now. who does she pick? i have to write 10 lines in order to reply to the first comment. what am i going to say. la da da de de. la da da de de nope only up to 8 how do i get to 9 almost almost awww 9 now i need 10 - 1, 2, 3, 4, 5, 6, 7, i missed counted this is only number 8. punky brewster is pretty awesome too. almost to 10 almost awwwwww.',
 "ok, to start with, this movie was not at all like the book. i read the book when i was 13 and since then it has always been my favorite. when i'm waiting for a different book to come out, this is the book i turn to to fill in the time. i have 3 copies for god's sake. anyways, i knew this was not going to be anything like the book but come on! they could h

In [26]:
def remove_urls(text):
    pattern = r'https?://\S+|www\S+'
    return re.sub(pattern, '', text)

In [27]:
df["clean_review"] = df["clean_review"].apply(remove_urls)

In [28]:
df["clean_review"].str.contains(
    r'https?://|www\.',
    case=False,
    regex=True
).sum()

np.int64(0)

# Step 5: Punctuation Removal

Punctuation does not usually contribute to sentiment prediction.

Removing punctuation reduces vocabulary size and improves feature quality.

In [29]:
print(string.punctuation)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [30]:
def remove_punctuation(text):
    return text.translate(
        str.maketrans(
            '',
            '',
            string.punctuation
        )
    )

In [31]:
text = "i loved this movie!!!"

remove_punctuation(text)

'i loved this movie'

In [32]:
df["clean_review"] = df["clean_review"].apply(
    remove_punctuation
)

In [33]:
print(df["review"][0][:300])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru


In [34]:
print(df["clean_review"][0][:300])

one of the other reviewers has mentioned that after watching just 1 oz episode youll be hooked they are right as this is exactly what happened with methe first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not 


In [35]:
import string

punct_count = df["clean_review"].str.contains(
    f"[{string.punctuation}]",
    regex=True
).sum()

print(punct_count)

0


# Step 5 Findings

### Punctuation Removal

The following punctuation symbols were removed:

- .
- ,
- !
- ?
- :
- ;
- '
- "
- (
- )
- /
- -

### Benefits

1. Reduced vocabulary size.
2. Improved text consistency.
3. Simplified tokenization.

### Conclusion

Punctuation noise has been removed successfully.

In [36]:
print(punct_count)

0


In [37]:
df["clean_review"].str.contains(
    r"\d",
    regex=True
).sum()

np.int64(27751)

# Step 6: Number Removal

Numbers generally do not contribute significantly to sentiment classification.

Removing numbers helps reduce noise and improves text consistency.

In [38]:
def remove_numbers(text):
    return re.sub(r"\d+", "", text)

In [39]:
df["clean_review"] = df["clean_review"].apply(
    remove_numbers
)

In [40]:
df["clean_review"].str.contains(
    r"\d",
    regex=True
).sum()

np.int64(0)

In [41]:
print(df["review"][0][:300])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru


In [42]:
print(df["clean_review"][0][:300])

one of the other reviewers has mentioned that after watching just  oz episode youll be hooked they are right as this is exactly what happened with methe first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not a


# Step 6 Findings

### Number Removal

Numeric values such as:

- 10
- 100
- 2007
- 3

were removed successfully.

### Benefits

1. Reduced textual noise.
2. Improved feature consistency.
3. Simplified vocabulary.

### Conclusion

Numbers have been removed from the review text.

In [43]:
type(df["clean_review"][0])

str

# Step 7: Tokenization
What is Tokenization?

Tokenization means:

Before:   
i love this movie

After:  
['i', 'love', 'this', 'movie']

These individual words are called:

Tokens

In [44]:
df["tokens"] = df["clean_review"].apply(
    lambda x: x.split()
)

In [45]:
df[
    ["clean_review", "tokens"]
].head()

,clean_review,tokens
0,one of the other reviewers has mentioned that ...,"[one, of, the, other, reviewers, has, mentione..."
1,a wonderful little production the filming tech...,"[a, wonderful, little, production, the, filmin..."
2,i thought this was a wonderful way to spend ti...,"[i, thought, this, was, a, wonderful, way, to,..."
3,basically theres a family where a little boy j...,"[basically, theres, a, family, where, a, littl..."
4,petter matteis love in the time of money is a ...,"[petter, matteis, love, in, the, time, of, mon..."


In [46]:
print(df["tokens"][0])

['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'youll', 'be', 'hooked', 'they', 'are', 'right', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'methe', 'first', 'thing', 'that', 'struck', 'me', 'about', 'oz', 'was', 'its', 'brutality', 'and', 'unflinching', 'scenes', 'of', 'violence', 'which', 'set', 'in', 'right', 'from', 'the', 'word', 'go', 'trust', 'me', 'this', 'is', 'not', 'a', 'show', 'for', 'the', 'faint', 'hearted', 'or', 'timid', 'this', 'show', 'pulls', 'no', 'punches', 'with', 'regards', 'to', 'drugs', 'sex', 'or', 'violence', 'its', 'is', 'hardcore', 'in', 'the', 'classic', 'use', 'of', 'the', 'wordit', 'is', 'called', 'oz', 'as', 'that', 'is', 'the', 'nickname', 'given', 'to', 'the', 'oswald', 'maximum', 'security', 'state', 'penitentary', 'it', 'focuses', 'mainly', 'on', 'emerald', 'city', 'an', 'experimental', 'section', 'of', 'the', 'prison', 'where', 'all', 'the', 'cells', 'have', 'glass

In [47]:
len(df["tokens"][0])

300

In [48]:
type(df["tokens"][0])

list

# Step 7 Findings

### Tokenization

The cleaned reviews were converted into lists of words (tokens).

Example:

Before:
i love this movie

After:
['i', 'love', 'this', 'movie']

### Benefits

1. Enables stopword removal.
2. Enables lemmatization.
3. Prepares text for feature extraction.

### Conclusion

The review text has been successfully tokenized.

# Step 8: Stopword Removal (Sentiment-Aware Version)
What are Stopwords?

Stopwords are extremely common words that usually do not carry significant meaning.

Examples:

the
a
an
is
are
was
were
of
in
on
at
for
with

Example
Before
['this', 'movie', 'is', 'very', 'good']
After
['movie', 'good']

In [49]:
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [50]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

In [51]:
sentiment_words = {
    "not",
    "no",
    "nor",
    "never"
}

In [52]:
stop_words = stop_words - sentiment_words

In [53]:
print("not" in stop_words)
print("no" in stop_words)
print("nor" in stop_words)
print("never" in stop_words)

False
False
False
False


In [54]:
def remove_stopwords(tokens):
    return [
        word
        for word in tokens
        if word not in stop_words
    ]

In [55]:
df["tokens_no_stopwords"] = df["tokens"].apply(
    remove_stopwords
)

In [56]:
print(df["tokens"][0][:30])

['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'youll', 'be', 'hooked', 'they', 'are', 'right', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'methe', 'first', 'thing', 'that']


In [57]:
print(df["tokens_no_stopwords"][0][:30])

['one', 'reviewers', 'mentioned', 'watching', 'oz', 'episode', 'youll', 'hooked', 'right', 'exactly', 'happened', 'methe', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scenes', 'violence', 'set', 'right', 'word', 'go', 'trust', 'not', 'show', 'faint', 'hearted', 'timid']


In [58]:
len(df["tokens"][0])

300

In [59]:
len(df["tokens_no_stopwords"][0])

171

In [60]:
test_sentence = [
    "this",
    "movie",
    "is",
    "not",
    "good"
]

remove_stopwords(test_sentence)

['movie', 'not', 'good']

# Step 8 Findings

### Stopword Removal

Common English stopwords were removed from the tokenized reviews.

Examples removed:

- the
- is
- are
- was
- of
- in
- on
- at

Important sentiment words retained:

- not
- no
- nor
- never

### Benefits

1. Reduced vocabulary size.
2. Reduced textual noise.
3. Improved sentiment preservation.

### Conclusion

Stopword removal was successfully applied while preserving important sentiment information.

In [61]:
print(df["tokens"][0][:30])
print(df["tokens_no_stopwords"][0][:30])

print(len(df["tokens"][0]))
print(len(df["tokens_no_stopwords"][0]))

['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'youll', 'be', 'hooked', 'they', 'are', 'right', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'methe', 'first', 'thing', 'that']
['one', 'reviewers', 'mentioned', 'watching', 'oz', 'episode', 'youll', 'hooked', 'right', 'exactly', 'happened', 'methe', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scenes', 'violence', 'set', 'right', 'word', 'go', 'trust', 'not', 'show', 'faint', 'hearted', 'timid']
300
171


# Step 9: Lemmatization

This is the final major NLP preprocessing step.

# What is Lemmatization?

Words can appear in many forms:

run
running
runs
ran

All represent the same concept.

Lemmatization converts them to their root form (lemma).

In [63]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [64]:
def lemmatize_tokens(tokens):
    text = " ".join(tokens)

    doc = nlp(text)

    return [
        token.lemma_
        for token in doc
    ]

In [65]:
df["tokens_lemmatized"] = df[
    "tokens_no_stopwords"
].apply(
    lemmatize_tokens
)

In [66]:
print(df["tokens_no_stopwords"][0][:30])

['one', 'reviewers', 'mentioned', 'watching', 'oz', 'episode', 'youll', 'hooked', 'right', 'exactly', 'happened', 'methe', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scenes', 'violence', 'set', 'right', 'word', 'go', 'trust', 'not', 'show', 'faint', 'hearted', 'timid']


In [67]:
print(df["tokens_lemmatized"][0][:30])

['one', 'reviewer', 'mention', 'watch', 'oz', 'episode', 'you', 'll', 'hook', 'right', 'exactly', 'happen', 'methe', 'first', 'thing', 'strike', 'oz', 'brutality', 'unflinche', 'scene', 'violence', 'set', 'right', 'word', 'go', 'trust', 'not', 'show', 'faint', 'hearted']


In [68]:
type(df["tokens_lemmatized"][0])

list

In [69]:
len(df["tokens_lemmatized"][0])

177

# Step 9 Findings

### Lemmatization

Words were converted to their root forms.

Examples:

reviewers → reviewer

mentioned → mention

watching → watch

movies → movie

### Benefits

1. Reduced vocabulary size.
2. Improved semantic consistency.
3. Better feature extraction for machine learning.

### Conclusion

Lemmatization was successfully applied using spaCy.

# Step 10: Final Preprocessing Pipeline

In [70]:
df["final_text"] = df[
    "tokens_lemmatized"
].apply(
    lambda tokens: " ".join(tokens)
)

In [71]:
print(df["final_text"][0])

one reviewer mention watch oz episode you ll hook right exactly happen methe first thing strike oz brutality unflinche scene violence set right word go trust not show faint hearted timid show pull no punch regard drug sex violence hardcore classic use wordit call oz nickname give oswald maximum security state penitentary focus mainly emerald city experimental section prison cell glass front face inward privacy not high agenda em city home manyaryan muslim gangstas latinos christians italians irish moreso scuffle death stare dodgy dealing shady agreement never far awayi would say main appeal show due fact go show would not dare forget pretty picture paint mainstream audience forget charm forget romanceoz do not mess around first episode ever see strike nasty surreal could not say ready watch develop taste oz get accustomed high level graphic violence not violence injustice crook guard who ll sell nickel inmate who ll kill order get away well mannered middle class inmate turn prison bitc

In [72]:
type(df["final_text"][0])

str

In [73]:
df[
    [
        "review",
        "clean_review",
        "final_text"
    ]
].head(1)

,review,clean_review,final_text
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one reviewer mention watch oz episode you ll h...


In [74]:
df.to_csv(
    "../data/processed/imdb_preprocessed.csv",
    index=False
)

In [75]:
import os

os.path.exists(
    "../data/processed/imdb_preprocessed.csv"
)

True

# Step 10 Findings

### Final NLP Pipeline

The following preprocessing steps were applied:

1. Lowercasing
2. HTML Tag Removal
3. URL Removal
4. Punctuation Removal
5. Number Removal
6. Tokenization
7. Stopword Removal
8. Lemmatization

### Final Output

A processed text column named `final_text` was created and saved.

### Conclusion

The dataset is now ready for feature extraction using TF-IDF and machine learning models.